# 05 · Phase 3: ungrounded LLM explanations

The detector (**EfficientNet-B0**, chosen in 04b) decides real/fake; Grad-CAM++ shows where its evidence is.
Our code turns that into a short **evidence summary** (fixed template, no AI). An **LLM** reads only that
summary (it never sees the image) and returns a JSON explanation: a facial **region** and an **artifact type**
from fixed lists, plus 2–4 plain sentences. This is the **ungrounded baseline**; phase 4 adds forensic
literature to the same prompt, and phase 5 compares the two.

Three models on Groq's free tier explain the same evidence: **GPT-OSS-120B** (main model) and **GPT-OSS-20B**,
the same OpenAI open-weight family at two sizes, and **Qwen3.8-27B** (Alibaba, a Groq preview model) as a second
maker. All three reason before answering; reasoning effort is set to *low* for every model.
The LLMs run on Groq's servers; this notebook's GPU only runs the detector and Grad-CAM++.
Only the text evidence summary is sent, never an image.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 · **Internet:** On
- **Inputs:** **01_extract_faces** (FF++), **01b_extract_celebdf**, **02_train** (the copy with EfficientNet-B0)
- **Secrets (Add-ons → Secrets):** `GROQ_API_KEY` (free key from console.groq.com).

**Free-tier limits (per model):** 30 requests and 8K tokens per minute, 1K requests and 200K tokens per day.
Requests are spaced 12 s apart (`MIN_INTERVAL` below) to stay under 8K tokens per minute, so each model takes
about 15 minutes. This pilot uses roughly 100K tokens per model; if a daily limit is hit, the remaining images
get an error row instead of an explanation.

**Produces:** `/kaggle/working/outputs/explanations/` (evidence table + every LLM answer, with prompts and raw replies).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic openai
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
%matplotlib inline
import inspect, json, os, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from src.kaggle_utils import setup, find_datasets, checkpoint_roots, setup_llm
setup(data=False)

from src.explain import providers
assert "groq" in providers.PROVIDERS, "Outdated code: restart the session (Run → Restart) and run all again."
from src.classifier.compare import find_checkpoints
from src.config import IMG_SIZE
from src.eval.metrics import faithfulness_iou, hallucination_flags, pointing_hit
from src.explain.corpus import artifact_taxonomy
from src.explain.llm import SYSTEM, REGIONS, ForensicExplainer
from src.xai.evaluate_cam import sample_images
from src.xai.gradcam import Explainer, draw_bbox, draw_regions

DATASETS = {k: v for k, v in find_datasets().items() if k in ("FF++", "CelebDF")}
CKPTS = find_checkpoints(checkpoint_roots())
print("datasets:", DATASETS)
print("detectors available:", list(CKPTS))
OUT = Path("/kaggle/working/outputs/explanations")
OUT.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
DETECTOR = "efficientnet_b0"          # chosen in notebook 04b
LLMS = [("groq", "openai/gpt-oss-120b"),      # (provider, model): OpenAI open weights, large (main model)
        ("groq", "openai/gpt-oss-20b"),       # same family, small
        ("groq", "qwen/qwen3.8-27b")]         # Alibaba Qwen (a preview model on Groq)
MIN_INTERVAL = {"groq": 12, "gemini": 6}      # seconds between requests, to stay under free-tier per-minute limits
os.environ["LLM_REASONING_EFFORT"] = "low"    # same reasoning effort for every model
RANK_BY = "concentration"             # rank regions by heat per pixel ("share" = total heat, favours big regions)
N_PER_DATASET = 25                    # pilot images per dataset (one frame per test video, balanced by method)
assert DETECTOR in CKPTS, f"{DETECTOR} not found: attach the 02_train copy that trained it"

## 1 · Evidence for each image
The detector and Grad-CAM++ run once per image; the same evidence summary is then given to every LLM.

In [ ]:
explainer = Explainer(CKPTS[DETECTOR], rank_by=RANK_BY)

items = []   # worked examples (one per manipulation per dataset) + pilot sample, from different videos
for ds, root in DATASETS.items():
    pilot = sample_images(Path(root), N_PER_DATASET)
    pilot_videos = set(pilot.video)
    by_method = {}
    for p in sorted(Path(root, "test").glob("*/*.jpg")):
        if "__".join(p.name.split("__")[:2]) not in pilot_videos:
            by_method.setdefault(p.name.split("__")[0], []).append(p)
    rng = np.random.default_rng(7)
    for method, ps in sorted(by_method.items()):
        p = ps[rng.integers(len(ps))]
        items.append({"set": "example", "dataset": ds, "path": str(p), "file": p.name, "manipulation": method,
                      "label": p.parent.name})
    for r in pilot.itertuples():
        items.append({"set": "pilot", "dataset": ds, "path": r.path, "file": r.file, "manipulation": r.method,
                      "label": r.label})

for it in items:
    it["exp"] = explainer(Image.open(it["path"]))
del explainer

evidence = pd.DataFrame([{k: v for k, v in it.items() if k != "exp"} | {
    "pred": it["exp"].label, "prob_fake": round(it["exp"].prob_fake, 4), "confidence": it["exp"].confidence,
    "top_regions": [r for r, _ in it["exp"].top_regions[:3]], "summary": it["exp"].describe()} for it in items])
evidence.to_csv(OUT / "evidence.csv", index=False)
print(evidence.groupby(["set", "dataset"]).size().to_string())
print(f"detector accuracy on these images: {(evidence.pred == evidence.label).mean():.0%}")
display(evidence[["set", "dataset", "manipulation", "label", "pred", "prob_fake", "confidence", "top_regions"]].head(10))

## 2 · Exactly what the LLM sees
Only this text, never the image, the true label, the dataset or the manipulation name.

In [ ]:
system_prompt = SYSTEM.format(regions=json.dumps(REGIONS), artifacts=json.dumps(sorted(artifact_taxonomy())))
display(Markdown("**System prompt (same for every image and every model):**"))
print(system_prompt)
display(Markdown("**User message (one per image), e.g. the first example:**"))
print(f"Evidence summary:\n{items[0]['exp'].describe()}")

## 3 · Run each LLM
Every model explains every image with the same prompts, one API request per image. A model whose API key is
missing, or that fails, is skipped and reported; a single failed request is kept as an error row for that image.

In [ ]:
results, run_log = {}, []
exps = [it["exp"] for it in items]
for provider, model_id in LLMS:
    print(f"\n===== {provider}: {model_id} =====")
    t0 = time.time()
    os.environ["LLM_MIN_INTERVAL"] = str(MIN_INTERVAL.get(provider, 0))
    try:
        if not setup_llm(provider, model_id):
            raise RuntimeError(f"add the {providers.PROVIDERS[provider].key_env} secret")
        llm = ForensicExplainer()
        outs = llm.explain_batch(exps, grounded=False)
        results[model_id] = outs
        rows = [{"llm": model_id, "provider": provider, **{k: v for k, v in it.items() if k not in ("exp", "path")},
                 **o.to_dict()} for it, o in zip(items, outs)]
        name = model_id.replace("/", "__")
        with open(OUT / f"phase3_{name}.jsonl", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in rows)
        run_log.append({"llm": model_id, "provider": provider, "status": "done",
                        "minutes": round((time.time() - t0) / 60, 1),
                        "schema_ok": f"{np.mean([o.schema_ok for o in outs]):.0%}",
                        "request_errors": sum(bool(o.error) and not o.raw for o in outs)})
        del llm
    except Exception as e:
        print(f"SKIPPED {model_id}: {type(e).__name__}: {e}")
        run_log.append({"llm": model_id, "provider": provider, "status": f"failed: {type(e).__name__}: {e}"[:120],
                        "minutes": round((time.time() - t0) / 60, 1)})
display(pd.DataFrame(run_log))
first_error = next((o.error for outs in results.values() for o in outs if o.error and not o.raw), "")
if first_error:
    print("first request error:", first_error[:300])
assert results, "No LLM produced explanations; see the errors above."

## 4 · Worked examples
For each example: the face, the detector's Grad-CAM++ evidence with the facial-region boxes, the evidence summary, and every LLM's explanation. **Checks** (computed automatically against the heatmap): region among the top-3, artifact plausible for those regions, artifact consistent with the verdict.

In [ ]:
SHORT_FLAGS = {"region_unsupported": "region not in top-3", "artifact_unsupported": "artifact implausible for the regions",
               "verdict_mismatch": "artifact contradicts verdict", "citation_invalid": "invented citation"}

def checks(o, exp):
    if not o.schema_ok:
        return "⚠️ invalid output" + (f" ({o.error[:60]})" if o.error else "")
    f = hallucination_flags(o, exp)
    bad = [SHORT_FLAGS[k] for k, v in f.items() if v and k in SHORT_FLAGS]
    return "✅ consistent with the evidence" if not bad else "❌ " + "; ".join(bad)

for i, it in enumerate(items):
    if it["set"] != "example":
        continue
    exp = it["exp"]
    rgb = np.asarray(Image.open(it["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    axes[0].imshow(rgb); axes[1].imshow(draw_bbox(exp.overlay, exp.bbox)); axes[2].imshow(draw_regions(rgb, exp.regions))
    for ax, t in zip(axes, ("face", "Grad-CAM++ evidence", "facial regions")):
        ax.set_title(t, fontsize=8); ax.axis("off")
    truth = "real" if it["manipulation"] == "original" else f"fake ({it['manipulation']})"
    plt.suptitle(f"{it['dataset']} · truth: {truth} · detector: {exp.label} ({exp.prob_fake:.3f}, {exp.confidence})", fontsize=9)
    plt.tight_layout(); plt.show()
    rows = "".join(f"| `{m.split('/')[-1]}` | {o.claimed_region or '—'} | `{o.artifact_type or '—'}` | "
                   f"{(o.explanation or o.error or o.raw[:120]).replace('|', '/')} | {checks(o, exp)} |\n"
                   for m, outs in results.items() for o in [outs[i]])
    display(Markdown(f"*Evidence summary:* {exp.describe()}\n\n| LLM | region | artifact | explanation | checks |\n"
                     f"|---|---|---|---|---|\n{rows}"))

## 5 · Pilot results
Pilot images only (the worked examples come from other videos and are not counted), so every image counts once.
**Valid output** = JSON with labels from the allowed lists. **Region in top-3** is easy to satisfy because the
prompt lists those regions; **pointing hit** (the named region contains the heatmap's hottest pixel) and
**region IoU** are the stricter checks. The rates preview phase 5's hallucination checks for the ungrounded condition.

In [ ]:
stats, artifacts = [], {}
for model_id, outs in results.items():
    rows = []
    for it, o in zip(items, outs):
        if it["set"] != "pilot":
            continue
        f = hallucination_flags(o, it["exp"]) if o.schema_ok else {}
        rows.append({"valid": o.schema_ok, "parsed": not o.error,
                     "region_in_top3": o.schema_ok and not f.get("region_unsupported", True),
                     "pointing_hit": o.schema_ok and pointing_hit(o.claimed_region, it["exp"].heatmap, it["exp"].regions),
                     "artifact_plausible": o.schema_ok and not f.get("artifact_unsupported", True),
                     "verdict_consistent": o.schema_ok and not f.get("verdict_mismatch", True),
                     "no_hallucination_flag": o.schema_ok and not f.get("any_hallucination", True),
                     "region_iou": faithfulness_iou(o.claimed_region, it["exp"].heatmap, regions=it["exp"].regions)
                                   if o.schema_ok else np.nan,
                     "latency_s": o.latency_s})
    r = pd.DataFrame(rows)
    stats.append({"llm": model_id.split("/")[-1], "n": len(r), **r.drop(columns=["latency_s", "region_iou"]).mean().round(3).to_dict(),
                  "mean_region_iou": round(r.region_iou.mean(), 3), "sec_per_explanation": round(r.latency_s.mean(), 1)})
    artifacts[model_id.split("/")[-1]] = pd.Series([o.artifact_type for it, o in zip(items, outs)
                                                    if it["set"] == "pilot" and o.schema_ok]).value_counts()
stats = pd.DataFrame(stats).set_index("llm")
stats.to_csv(OUT / "phase3_pilot_summary.csv")
display(stats)

rates = ["valid", "region_in_top3", "pointing_hit", "artifact_plausible", "verdict_consistent", "no_hallucination_flag"]
ax = stats[rates].T.plot.bar(figsize=(11, 3.6), rot=0, width=0.8, ylim=(0, 1.05))
ax.set_ylabel("share of explanations"); ax.set_title("Phase 3 pilot: ungrounded explanations (higher is better)", loc="left", fontsize=10)
ax.grid(axis="y", color="#d9d9d4", linewidth=0.6); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / "phase3_pilot.png", dpi=150); plt.show()

display(Markdown("**Which artifact types each LLM chose** (a model that always picks the same one is not reading the evidence)"))
display(pd.DataFrame(artifacts).fillna(0).astype(int))
print("saved:", sorted(p.name for p in OUT.iterdir()))